In [0]:
raw_path = "s3://smart-traffic-analytics/landing/traffic_events"

bronze_path = "s3://smart-traffic-analytics/bronze/traffic_events"

checkpoint_path = "s3://smart-traffic-analytics/checkpoint/bronze"

schema_path = "s3://smart-traffic-analytics/schema/bronze"

In [0]:
from pyspark.sql.types import *

traffic_schema = StructType([
    StructField("event_id", IntegerType(), True),
    StructField("road_name", StringType(), True),
    StructField("vehicle_count", IntegerType(), True),
    StructField("avg_speed", IntegerType(), True),
    StructField("weather", StringType(), True),
    StructField("accident_status", BooleanType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("timestamp", StringType(), True)
])


In [0]:
traffic_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", schema_path)
        .schema(traffic_schema)
        .load(raw_path)
)

In [0]:
bronze_df = (
    traffic_df
    .withColumn("ingestion_time", current_timestamp())
    .withColumn("source_system", lit("Python Traffic Simulator"))
)

In [0]:
query = (
    bronze_df.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .outputMode("append")
        .trigger(availableNow=True)
        .start(bronze_path)
)

query.awaitTermination()

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS bronze_traffic_events
USING DELTA
LOCATION '{bronze_path}'
""")

In [0]:
display(
    spark.sql("""
    SELECT *
    FROM bronze_traffic_events
    """)
)

In [0]:
spark.sql("""
SELECT COUNT(*) AS total_records
FROM bronze_traffic_events
""").show()